# 小米 14：第二次初版配对尝试

## tl;dr

0/48 个进程取得翻译样本；在等待起跑条件时，前台核验失败。不是翻译崩溃，也不是频率门槛超时。

## Context & Methods

### Key Assumptions

v0.1.0 与本地 v0.3.0，固定八场景、三轮、每进程三遍。沿用第一次构建及协议；场次单独复核，不拼接或将缺失性能填零。

## Data

本次原始记录 `attempt-2.json`；前次 `attempt-1.json` 仅用于核对构建、输入及协议一致性。

In [1]:
from pathlib import Path
import json, runpy
root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'tools/version-bench/check.py').is_file())
record = root / 'benchmarks/v0.3.0/mi14-2026-09-09/initial-to-current'
checks = runpy.run_path(str(record / 'verify.py'))
result = checks['verify']('attempt-2.json')
raw = json.loads((record / 'attempt-2.json').read_text())
prior = json.loads((record / 'attempt-1.json').read_text())
for key in ('versions', 'binary_sha256', 'harness_sha256', 'model_files_sha256', 'corpus_sha256', 'protocol', 'suite_sha256', 'config_yaml'):
    assert raw[key] == prior[key], key

## Results

### 完整性与退出原因

In [2]:
assert not result['accepted'] and result['exit_code'] == 2
assert result['successful_processes'] == 0 and result['planned_processes'] == 48
assert len(result['rows']) == 16
assert all(not r['complete'] and 'cold_inputs_per_second' not in r for r in result['rows'])
assert len(raw['runs']) == 1 and raw['runs'][0]['samples'] == []
assert raw['runs'][0]['before']['gate_wait_s'] == -2
print({k:v for k,v in result.items() if k != 'rows'})
print(json.dumps(raw['runs'][0], ensure_ascii=False, indent=2))

{'file': 'attempt-2.json', 'accepted': False, 'exit_code': 2, 'successful_processes': 0, 'planned_processes': 48, 'diagnostic': 'INVALID: v0.1.0: missing or unexpected scenarios'}
{
  "round": 1,
  "scenario": "enzh_w1",
  "version": "v0.1.0",
  "before": {
    "max_freq_khz": {
      "2": "2572800",
      "7": "1939200"
    },
    "battery_temp_c": 33.7,
    "board_temp": "34598",
    "foreground": {
      "wakefulness": "Awake",
      "lockscreen": false,
      "focused_package": null
    },
    "gate_wait_s": -2
  },
  "returncode": 2,
  "samples": [],
  "error": "device not awake/unlocked with benchmark app foreground"
}


## Takeaways

没有可用于 README 的小米 14 初版对照数据。前台包名为空不等于已证明 app 崩溃。CPU 亲和性与被监测策略范围存在不一致，留待独立修正；本次不修改采集协议。